# Comparação da média diária da temperatura

Este notebook calcula e sobrepõe as médias diárias da temperatura das bases tratadas da **NASA POWER** e da **SONDA-MD**.

- Período: **05/01/2024 a 05/01/2026**, com as duas datas incluídas;
- eixo horizontal: data;
- eixo vertical: temperatura em °C;
- a média diária é calculada diretamente a partir dos valores horários válidos de cada fonte.

In [ ]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 130
plt.rcParams['savefig.dpi'] = 300

INICIO = pd.Timestamp('2024-01-05 00:00:00')
FIM = pd.Timestamp('2026-01-05 23:59:59')
ARQUIVO_NASA = 'nasa_power_horario_padronizado.csv'
ARQUIVO_SONDA = 'sonda_md_horario_padronizado.csv'
VALORES_INVALIDOS = [-9999, -999, -6999, 7999, 9999]

def encontrar_pasta_dados():
    """Encontra a pasta 2026/dados_tratados a partir do diretório de execução."""
    inicio = Path.cwd().resolve()
    for pasta in (inicio, *inicio.parents):
        candidata = pasta / '2026' / 'dados_tratados'
        if (candidata / ARQUIVO_NASA).exists() and (candidata / ARQUIVO_SONDA).exists():
            return candidata
    raise FileNotFoundError(
        'Não foi encontrada a pasta 2026/dados_tratados com as duas bases tratadas.'
    )

PASTA_DADOS = encontrar_pasta_dados()
PASTA_SAIDA = PASTA_DADOS.parent / 'resultados' / 'figuras' / 'comparacao_bases'
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

print(f'Pasta de dados: {PASTA_DADOS}')
print(f'Período selecionado: {INICIO:%d/%m/%Y} a {FIM:%d/%m/%Y}')

## Leitura e recorte das bases horárias

As fontes são lidas separadamente. O recorte usa o intervalo completo solicitado, sem interpolar ou criar observações.

In [ ]:
def ler_temperatura(caminho):
    dados = pd.read_csv(caminho, parse_dates=['data_hora'])
    dados['temperatura'] = pd.to_numeric(dados['temperatura'], errors='coerce')
    dados['temperatura'] = dados['temperatura'].replace(VALORES_INVALIDOS, np.nan)
    dados = (
        dados[['data_hora', 'temperatura']]
        .dropna(subset=['data_hora'])
        .sort_values('data_hora')
        .drop_duplicates('data_hora')
    )
    return dados.loc[
        (dados['data_hora'] >= INICIO) & (dados['data_hora'] <= FIM)
    ].copy()

nasa_horaria = ler_temperatura(PASTA_DADOS / ARQUIVO_NASA)
sonda_md_horaria = ler_temperatura(PASTA_DADOS / ARQUIVO_SONDA)

print(f'NASA POWER: {len(nasa_horaria):,} registros horários')
print(f'SONDA-MD:   {len(sonda_md_horaria):,} registros horários')
display(pd.DataFrame({
    'fonte': ['NASA POWER', 'SONDA-MD'],
    'primeiro horário': [nasa_horaria['data_hora'].min(), sonda_md_horaria['data_hora'].min()],
    'último horário': [nasa_horaria['data_hora'].max(), sonda_md_horaria['data_hora'].max()],
    'valores ausentes de temperatura': [nasa_horaria['temperatura'].isna().sum(), sonda_md_horaria['temperatura'].isna().sum()],
}))

## Cálculo das médias diárias

A média é calculada por fonte, agrupando os registros horários pela data. A tabela resultante mantém uma linha por dia do período.

In [ ]:
def calcular_media_diaria(dados):
    serie = dados.set_index('data_hora')['temperatura']
    return serie.resample('D').agg(media_temperatura='mean', observacoes_horarias='count')

diaria_nasa = calcular_media_diaria(nasa_horaria).rename(
    columns={'media_temperatura': 'temperatura_nasa_power', 'observacoes_horarias': 'n_horarios_nasa_power'}
)
diaria_sonda = calcular_media_diaria(sonda_md_horaria).rename(
    columns={'media_temperatura': 'temperatura_sonda_md', 'observacoes_horarias': 'n_horarios_sonda_md'}
)

medias_diarias = diaria_nasa.join(diaria_sonda, how='outer')
medias_diarias.index.name = 'data'

quantidade_dias = (FIM.normalize() - INICIO.normalize()).days + 1
assert len(medias_diarias) == quantidade_dias, (len(medias_diarias), quantidade_dias)
assert medias_diarias[['temperatura_nasa_power', 'temperatura_sonda_md']].notna().all().all()

print(f'Dias representados: {len(medias_diarias):,}')
print('Menor e maior número de observações horárias por dia:')
display(medias_diarias[['n_horarios_nasa_power', 'n_horarios_sonda_md']].agg(['min', 'max']))
display(medias_diarias.head())

## Gráfico comparativo

As duas curvas são apresentadas no mesmo eixo vertical, em °C, para facilitar a comparação da evolução temporal.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(
    medias_diarias.index,
    medias_diarias['temperatura_nasa_power'],
    color='#2166ac',
    linewidth=1.35,
    label='NASA POWER',
)
ax.plot(
    medias_diarias.index,
    medias_diarias['temperatura_sonda_md'],
    color='#b2182b',
    linewidth=1.35,
    label='SONDA-MD',
)

ax.set_title('Média diária da temperatura: NASA POWER e SONDA-MD', fontsize=14, pad=12)
ax.set_xlabel('Data')
ax.set_ylabel('Temperatura (°C)')
ax.set_xlim(INICIO.normalize(), FIM.normalize())
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%Y'))
ax.legend(title='Fonte', frameon=True)
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
fig.tight_layout()

arquivo_figura = PASTA_SAIDA / 'comparacao_media_diaria_temperatura_nasa_power_sonda_md.png'
arquivo_tabela = PASTA_SAIDA / 'medias_diarias_temperatura_nasa_power_sonda_md.csv'
fig.savefig(arquivo_figura, bbox_inches='tight')
medias_diarias.to_csv(arquivo_tabela, date_format='%Y-%m-%d')
print(f'Figura salva em: {arquivo_figura}')
print(f'Tabela salva em: {arquivo_tabela}')
plt.show()